In [ ]:
import requests
import json
from notebookutils import mssparkutils

# --- Configuration ---
# API Base URL
fabric_api_base = "https://api.fabric.microsoft.com/v1"

# Auth Details (Update with your SPN or Key Vault logic)
tenant_id = "your-tenant-id"
client_id = "your-client-id"
client_secret = "your-client-secret" 
# Recommended: Use Key Vault
# client_secret = mssparkutils.credentials.getSecret("your-kv", "your-secret")

# --- Parameters ---
# Replace with actual IDs you want to DELETE
workspace_id = "your-workspace-id"
item_id = "your-item-id"

In [ ]:
def get_spn_token(scope):
    url = f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token"
    payload = {
        'grant_type': 'client_credentials',
        'client_id': client_id,
        'client_secret': client_secret,
        'scope': scope
    }
    try:
        response = requests.post(url, data=payload)
        response.raise_for_status()
        return response.json().get("access_token")
    except Exception as e:
        print(f"Failed to retrieve SPN token for scope {scope}: {e}")
        return None

# Get Fabric Token (for Delete Item API)
# Use Power BI Scope if Fabric scope fails: "https://analysis.windows.net/powerbi/api/.default"
fabric_scope = "https://api.fabric.microsoft.com/.default" 
fabric_token = get_spn_token(fabric_scope)

if fabric_token:
    print("Successfully retrieved Fabric token.")
else:
    print("Failed to retrieve Fabric token.")

In [ ]:
# --- Delete Item Function ---
def delete_fabric_item(workspace_id, item_id):
    """
    Deletes a specific item from a Fabric Workspace.
    API: DELETE https://api.fabric.microsoft.com/v1/workspaces/{workspaceId}/items/{itemId}
    """
    if not fabric_token:
        print("Skipping Delete Item: No Fabric token available.")
        return False

    url = f"{fabric_api_base}/workspaces/{workspace_id}/items/{item_id}"
    
    headers = {
        "Authorization": f"Bearer {fabric_token}",
        "Content-Type": "application/json"
    }
    
    print(f"Attempting to DELETE item {item_id} from workspace {workspace_id}...")
    
    try:
        response = requests.delete(url, headers=headers)
        
        # 200 OK or 204 No Content are typical success codes for DELETE
        if response.status_code in [200, 204]:
            print(f"SUCCESS: Deleted item {item_id} from workspace {workspace_id}.")
            return True
        elif response.status_code == 404:
            print(f"SKIPPED: Item {item_id} not found (already deleted?).")
            return True
        elif response.status_code == 429:
             print(f"FAILED (Rate Limit): {response.text}")
             return False
        else:
            print(f"FAILED: Could not delete item {item_id}. Status: {response.status_code}, Response: {response.text}")
            return False
            
    except Exception as e:
        print(f"EXCEPTION: Error deleting item {item_id}: {e}")
        return False

# --- Main Execution ---
if workspace_id and item_id and workspace_id != "your-workspace-id":
    # Confirm before running
    print(f"WARNING: This will PERMANENTLY DELETE item {item_id} from workspace {workspace_id}.")
    
    # Uncomment next line to run
    delete_fabric_item(workspace_id, item_id)
else:
    print("Please set valid workspace_id and item_id parameters in the first cell.")